# Deshimula All-Company Scraper

Extends the 2-company scraper to the **entire Deshimula company directory** (`/companies`, ~808 companies, ~2,466 reviews).

## Strategy (tiered)

1. **Directory crawl**: fetch `/companies?page={1..27}` and record every company's slug + its Deshi Mula review count (shown on the card). Companies with **0 reviews are skipped** entirely.
2. **Tier 1** (`>= 10` reviews, default): ~51 companies covering ~80% of all reviews. High quality-per-request.
3. **Tier 2** (`1..9` reviews): the long tail of ~670 small companies. Set `MIN_REVIEWS = 1` to include it.
4. **companyId resolution**: each company's profile page (`/companies/{slug}`) exposes its internal `companyId` and the **real company name** (`<h1>`).
5. **Per company**: paginate `/stories/{page}?companyId={id}&Vibe={1|2|3}`, scrape card metadata (role/date/votes/comments), then fetch each `/story/{id}` detail page for the full content.
6. **Rate limit + retry**: 0.3s between requests with retry/backoff on failure (Cloudflare friendly).
7. **Resumable**: progress is saved to `state.json` + the CSV, so re-running skips finished companies and never duplicates.

In [8]:
import csv
import json
import random
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://deshimula.com"
OUT_DIR = Path.cwd()

VIBES = {1: "Positive", 2: "Negative", 3: "Mixed"}

COMPANIES_DIR_PAGES = 27       # /companies has 27 pages of 30
# MIN_REVIEWS = 10               # Tier 1 threshold; set to 1 for full site
MIN_REVIEWS = 1

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

REQ_DELAY = 0.3               # min seconds between requests
MAX_RETRIES = 4
RETRY_BASE = 2.0              # exponential backoff base (seconds)

COMPANIES_CSV = OUT_DIR / "deshimula_companies_new.csv"
STATE_FILE = OUT_DIR / "state_new.json"
REVIEWS_CSV = OUT_DIR / "deshimula_all_reviews_new.csv"

FIELDS = [
    "id", "company_name", "developer_role", "date", "title",
    "content", "upvote", "downvote", "comment_count", "vibe",
]

In [9]:
class Client:
    """Rate-limited requests.Session with retry/backoff."""

    def __init__(self, delay: float = REQ_DELAY):
        self.session = requests.Session()
        self.session.headers.update(HEADERS)
        self.delay = delay
        self._last = 0.0

    def _throttle(self):
        wait = self.delay - (time.time() - self._last)
        if wait > 0:
            time.sleep(wait)

    def get(self, url: str) -> requests.Response:
        for attempt in range(MAX_RETRIES):
            self._throttle()
            try:
                resp = self.session.get(url, timeout=20)
                self._last = time.time()
                if resp.status_code == 200 and "Just a moment" not in resp.text[:3000]:
                    return resp
                # Cloudflare challenge or soft-4xx: treat as retryable
                if resp.status_code in (403, 429) or "Just a moment" in resp.text[:3000]:
                    raise requests.ConnectionError(f"challenge/limited: {resp.status_code}")
                resp.raise_for_status()
            except requests.RequestException as exc:
                backoff = RETRY_BASE * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"    retry {attempt + 1}/{MAX_RETRIES} after {backoff:.1f}s: {url} -> {exc}")
                time.sleep(backoff)
        raise RuntimeError(f"Failed after {MAX_RETRIES} retries: {url}")

In [10]:
def crawl_companies(client: Client) -> list[dict]:
    """Enumerate all companies from the /companies directory with their DM review counts."""
    companies = {}
    for page in range(1, COMPANIES_DIR_PAGES + 1):
        resp = client.get(f"{BASE_URL}/companies?page={page}")
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if not (href.startswith("/companies/") and href.count("/") == 2):
                continue
            slug = href.rsplit("/", 1)[-1]
            # walk up to the card block containing the review summary
            node = a
            for _ in range(30):
                node = node.parent
                if node is None:
                    break
                text = node.get_text(" ", strip=True)
                if "Positive" in text and "Negative" in text and "Deshi Mula" in text:
                    m = re.search(r"Deshi Mula\s+(\d+)", text)
                    count = int(m.group(1)) if m else 0
                    companies.setdefault(slug, {"slug": slug, "dm_reviews": count})
                    break
        print(f"  directory page {page}: {len(companies)} unique companies so far")
    return list(companies.values())

In [11]:
def resolve_company(client: Client, slug: str) -> dict:
    """Get the real name + companyId from a company profile page.

    Returns company_id=None when the page exposes no companyId (rare companies
    that must be scraped through the SearchTerm fallback instead).
    """
    resp = client.get(f"{BASE_URL}/companies/{slug}")
    soup = BeautifulSoup(resp.text, "html.parser")
    h1 = soup.find("h1")
    name = h1.get_text(strip=True) if h1 else slug.replace("-", " ").title()
    m = re.search(r"/stories/1\?companyId=([0-9a-f]{24})", resp.text)
    return {"company_name": name, "company_id": m.group(1) if m else None, "slug": slug}

In [12]:
def parse_card(card) -> dict:
    """Extract fields from one listing-card container."""
    link = next(
        a
        for a in card.find_all("a", href=True)
        if re.match(r"/story/[0-9a-f]{24}", a["href"]) and "Read" in a.get_text()
    )
    story_id = link["href"].split("/")[-1]
    h2 = card.find("h2")
    title = h2.get_text(" ", strip=True) if h2 else ""
    meta = card.find("div", class_=re.compile("mt-0.5"))
    role = date = ""
    if meta:
        spans = [s.get_text(strip=True) for s in meta.find_all("span")]
        spans = [s for s in spans if s]
        if len(spans) >= 3:
            role, _, date = spans[0], spans[1], spans[2]
    content_div = card.find("div", class_=re.compile("text-slate-800"))
    snippet = content_div.get_text(" ", strip=True) if content_div else ""
    votes_cont = None
    for div in card.find_all("div"):
        if div.find(string=re.compile("Upvotes")):
            votes_cont = div
            break
    nums = []
    if votes_cont:
        nums = [
            s.get_text(strip=True)
            for s in votes_cont.find_all("span")
            if s.get_text(strip=True).isdigit()
        ][-3:]
    nums = (nums + ["0"] * 3)[:3]
    return {
        "story_id": story_id,
        "title": title,
        "role": role,
        "date": date,
        "content": snippet,
        "upvote": int(nums[0]),
        "downvote": int(nums[1]),
        "comment_count": int(nums[2]),
    }

In [13]:
def scrape_story(client: Client, story_id: str) -> str:
    """Full review text from the /story/{id} detail page."""
    resp = client.get(f"{BASE_URL}/story/{story_id}")
    soup = BeautifulSoup(resp.text, "html.parser")
    content = soup.find("div", class_="story-content")
    return content.get_text("\n", strip=True) if content else ""


def scrape_company(client: Client, company: dict) -> list[dict]:
    """All reviews for one company (all vibes, all pages)."""
    rows = []
    for vibe_val, vibe_label in VIBES.items():
        page = 1
        while True:
            url = f"{BASE_URL}/stories/{page}?companyId={company['company_id']}&Vibe={vibe_val}"
            soup = BeautifulSoup(client.get(url).text, "html.parser")
            cards = []
            for link in soup.find_all("a", href=True):
                if re.match(r"/story/[0-9a-f]{24}", link["href"]) and "Read" in link.get_text():
                    card = link.find_parent("div", class_="container")
                    if card is not None:
                        cards.append(card)
            if not cards:
                break
            for card in cards:
                row = parse_card(card)
                full = scrape_story(client, row["story_id"])
                if full:
                    row["content"] = full
                rows.append({**row, "vibe": vibe_label})
            page += 1
    return rows


def scrape_company_by_search(client: Client, company: dict) -> list[dict]:
    """SearchTerm fallback for companies that expose no companyId.

    SearchTerm matches are fuzzy, so we keep only cards whose company link
    points back at our target slug. Everything is returned on page 1, but we
    still probe page 2+ defensively (it is always empty in practice).
    """
    slug = company["slug"]
    rows = []
    for vibe_val, vibe_label in VIBES.items():
        page = 1
        while True:
            term = slug.replace("-", " ")
            url = f"{BASE_URL}/stories/{page}?SearchTerm={term.replace(' ', '+')}&Vibe={vibe_val}"
            soup = BeautifulSoup(client.get(url).text, "html.parser")
            cards = []
            for link in soup.find_all("a", href=True):
                if not (re.match(r"/story/[0-9a-f]{24}", link["href"]) and "Read" in link.get_text()):
                    continue
                card = link.find_parent("div", class_="container")
                if card is None:
                    continue
                # only keep reviews belonging to our target company
                company_links = {
                    a["href"].rstrip("/")
                    for a in card.find_all("a", href=True)
                    if a["href"].startswith("/companies/")
                }
                if f"/companies/{slug}" not in company_links:
                    continue
                cards.append(card)
            if not cards:
                break
            for card in cards:
                row = parse_card(card)
                full = scrape_story(client, row["story_id"])
                if full:
                    row["content"] = full
                rows.append({**row, "vibe": vibe_label})
            page += 1
    return rows

In [14]:
def load_state() -> dict:
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {"done_slugs": [], "total_ids": 0}


def save_state(state: dict):
    STATE_FILE.write_text(json.dumps(state, indent=2))


def main():
    client = Client()
    state = load_state()
    done = set(state["done_slugs"])

    # 1. enumerate directory
    companies = crawl_companies(client)
    companies = [c for c in companies if c["dm_reviews"] >= MIN_REVIEWS]
    companies.sort(key=lambda c: c["dm_reviews"], reverse=True)
    print(f"companies with >= {MIN_REVIEWS} reviews: {len(companies)}")
    with COMPANIES_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["slug", "dm_reviews"])
        w.writeheader()
        w.writerows(companies)

    # 2. per-company fetch
    new = 0
    fresh = not REVIEWS_CSV.exists() or REVIEWS_CSV.stat().st_size == 0
    with REVIEWS_CSV.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if fresh:
            writer.writeheader()
        for i, company in enumerate(companies, 1):
            slug = company["slug"]
            if slug in done:
                print(f"  [{i}/{len(companies)}] skip {slug} (done)")
                continue
            try:
                info = resolve_company(client, slug)
                if info["company_id"]:
                    rows = scrape_company(client, info)
                else:
                    rows = scrape_company_by_search(client, info)
                    print(f"  [{i}/{len(companies)}] (fallback SearchTerm) {slug}")
            except Exception as exc:
                print(f"  [{i}/{len(companies)}] FAILED {slug}: {exc}")
                continue
            for row in rows:
                new += 1
                row["id"] = state["total_ids"] + new
                row["company_name"] = info["company_name"]
                writer.writerow({k: row.get(k, "") for k in FIELDS})
            f.flush()
            done.add(slug)
            state["done_slugs"] = sorted(done)
            state["total_ids"] += len(rows)
            save_state(state)
            print(f"  [{i}/{len(companies)}] {info['company_name']}: {len(rows)} reviews (total {state['total_ids']})")

    print(f"done. {new} new reviews this run. total {state['total_ids']}.")


if __name__ == "__main__":
    main()

  directory page 1: 30 unique companies so far
  directory page 2: 60 unique companies so far
  directory page 3: 90 unique companies so far
  directory page 4: 120 unique companies so far
  directory page 5: 150 unique companies so far
  directory page 6: 179 unique companies so far
  directory page 7: 209 unique companies so far
  directory page 8: 238 unique companies so far
  directory page 9: 268 unique companies so far
  directory page 10: 298 unique companies so far
  directory page 11: 328 unique companies so far
  directory page 12: 358 unique companies so far
  directory page 13: 387 unique companies so far
  directory page 14: 417 unique companies so far
  directory page 15: 446 unique companies so far
  directory page 16: 476 unique companies so far
  directory page 17: 506 unique companies so far
  directory page 18: 536 unique companies so far
  directory page 19: 565 unique companies so far
  directory page 20: 595 unique companies so far
  directory page 21: 623 unique 